# 00 — Visuals

Motivational figures for the QAOA × Markowitz portfolio report.
All figures are 12×6 and saved as PDFs to `plots/visuals/` (Drive on Colab,
local repo otherwise). Colour palette is loaded from `palette/palette.json`
(shared with `snp.py`).

> **Runtime:** ~30 s on Colab CPU.

### SETUP

In [ ]:
# === Bootstrap (Colab + local) ===
import os, urllib.request as _u
exec((open('../scripts/bootstrap.py') if os.path.exists('../scripts/bootstrap.py') else _u.urlopen('https://raw.githubusercontent.com/egil10/fys5419/main/project2/code/scripts/bootstrap.py')).read())

# === Project imports ===
import json
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

from scripts.colab     import out_dir
from scripts.snp       import SNP, apply_style, title
from scripts.baskets   import get
from scripts.portfolio import PortfolioProblem

# Shared palette — single source of truth lives in palette/palette.json
PALETTE = json.loads(open('../palette/palette.json', encoding='utf-8').read())
apply_style()

PLOTS_DIR = out_dir('plots', 'visuals')
print(f'Plots will be saved to: {PLOTS_DIR}')

FIGSIZE = (12, 6)
BASKET  = 'mag7'           # n=4 — small enough for clean visuals
START   = '2023-01-01'
END     = '2025-12-31'

### DATA

In [ ]:
snp = SNP(get(BASKET), START, END).cached_fetch(name=BASKET)
mu, Sigma   = snp.mu, snp.Sigma
mu_a, Sig_a = snp.annualised()
sigma_a     = np.sqrt(np.diag(Sig_a))
n           = snp.n
tickers     = snp.tickers
print(f'{BASKET}: {tickers}  n={n}  2^n={2**n}')

### 1. MARKOWITZ EFFICIENT FRONTIER

Classical mean-variance with continuous weights w in R^n, sum w = 1.
Random portfolios sampled from the simplex form the cloud; the red curve is the
analytical efficient frontier.

In [ ]:
# Random portfolios on the simplex (background cloud)
rng = np.random.default_rng(0)
W   = rng.dirichlet(np.ones(n), size=4000)
ret = W @ mu_a
vol = np.sqrt(np.einsum('ij,jk,ik->i', W, Sig_a, W))

# Analytical frontier (unconstrained, sum w = 1)
ones = np.ones(n)
Sinv = np.linalg.inv(Sig_a)
A    = ones @ Sinv @ ones
B    = ones @ Sinv @ mu_a
C    = mu_a  @ Sinv @ mu_a
D    = A * C - B ** 2
r_grid   = np.linspace(ret.min() * 0.4, ret.max() * 1.4, 400)
vol_grid = np.sqrt((A * r_grid ** 2 - 2 * B * r_grid + C) / D)

r_mvp, v_mvp = B / A, 1 / np.sqrt(A)
r_tan, v_tan = C / B, np.sqrt(C) / B

eff = r_grid >= r_mvp

fig, ax = plt.subplots(figsize=FIGSIZE)

# Cloud of random portfolios — restrained warm grey
ax.scatter(vol * 100, ret * 100,
           s=6, color=PALETTE['warm_grey'], alpha=0.35,
           edgecolor='none', label='Random portfolios')

# Frontier: efficient half bold, inefficient half thin dashed
ax.plot(vol_grid[eff]  * 100, r_grid[eff]  * 100,
        color=PALETTE['red'], lw=2.4, label='Efficient frontier', zorder=3)
ax.plot(vol_grid[~eff] * 100, r_grid[~eff] * 100,
        color=PALETTE['red'], lw=1.2, ls='--', alpha=0.55,
        label='Inefficient half', zorder=3)

# Capital allocation line (rf = 0) through the tangency portfolio
cal_x = np.array([0.0, v_tan * 1.45]) * 100
cal_y = (r_tan / v_tan) * cal_x
ax.plot(cal_x, cal_y, color=PALETTE['ochre'], lw=1.3, ls=':',
        label=f'Capital allocation line (Sharpe = {r_tan/v_tan:.2f})', zorder=2)

# Individual assets — filled circles in the per-asset palette
for k, t in enumerate(tickers):
    c = snp.colors[k]
    ax.scatter(sigma_a[k] * 100, mu_a[k] * 100,
               s=85, color=c, edgecolor='white', linewidth=1.0, zorder=5)
    ax.annotate(t, (sigma_a[k] * 100, mu_a[k] * 100),
                xytext=(8, 5), textcoords='offset points',
                fontsize=10, color=PALETTE['charcoal'], zorder=6)

# MVP and Tangency — small open markers with text labels
for x, y, label in [
    (v_mvp * 100, r_mvp * 100, 'Min variance'),
    (v_tan * 100, r_tan * 100, 'Max Sharpe'),
]:
    ax.scatter(x, y, s=80, facecolor='white',
               edgecolor=PALETTE['charcoal'], linewidth=1.5, zorder=6)
    ax.annotate(f'{label}\n({x:.1f}%, {y:.1f}%)', (x, y),
                xytext=(10, -4), textcoords='offset points',
                fontsize=9, color=PALETTE['charcoal'],
                fontweight='bold' if label == 'Max Sharpe' else 'normal',
                zorder=7)

ax.set_xlabel('Annualised volatility (%)')
ax.set_ylabel('Annualised return (%)')
title(ax, 'Markowitz efficient frontier',
      f'Continuous weights on the simplex — {BASKET} ({n} assets, daily data {START} to {END})')
ax.legend(loc='lower right', fontsize=9, frameon=False)
ax.set_xlim(left=0)
ax.grid(False)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'markowitz_frontier.pdf', bbox_inches='tight')
plt.show()

### 2. THE CARDINALITY CONSTRAINT MAKES IT DISCRETE

Restricting to binary selections (asset in or out, equal-weighted within the set)
turns the problem from a smooth curve into a finite set of 2^n - 1 points.
Selecting exactly K of n assets is the QUBO instance we hand to QAOA.

In [ ]:
# Every non-empty binary selection, equal-weighted within the selection
bitstrings = [b for b in product([0, 1], repeat=n) if sum(b) > 0]
W_disc = np.array([np.array(b) / sum(b) for b in bitstrings])
ret_d  = W_disc @ mu_a
vol_d  = np.sqrt(np.einsum('ij,jk,ik->i', W_disc, Sig_a, W_disc))
size   = np.array([sum(b) for b in bitstrings])

K_palette = [PALETTE['sky'], PALETTE['blue_muted'], PALETTE['blue'],
             PALETTE['navy'], PALETTE['crimson']]

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(vol_grid * 100, r_grid * 100, color=PALETTE['grid'], lw=2,
        label='Continuous frontier', zorder=1)
for k in range(1, n + 1):
    mask = size == k
    if mask.any():
        ax.scatter(vol_d[mask] * 100, ret_d[mask] * 100,
                   s=180, color=K_palette[min(k - 1, len(K_palette) - 1)],
                   edgecolor=PALETTE['charcoal'], linewidth=0.8,
                   label=f'K={k}  ({mask.sum()})', zorder=4)
for b, v, r in zip(bitstrings, vol_d, ret_d):
    ax.annotate(''.join(map(str, b)), (v * 100, r * 100),
                xytext=(6, 4), textcoords='offset points',
                fontsize=7, color=PALETTE['charcoal'], alpha=0.75)
ax.set_xlabel('Annualised volatility (%)')
ax.set_ylabel('Annualised return (%)')
title(ax, 'The cardinality constraint makes the problem discrete',
      f'Binary equal-weighted selections only — a finite set of $2^{{{n}}}-1={2**n - 1}$ points')
ax.legend(loc='best', fontsize=9, title='Selection size')
ax.grid(False)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'cardinality_constrained.pdf', bbox_inches='tight')
plt.show()

### 3. THE COMBINATORIAL WALL

Brute force enumerates all 2^n bitstrings. At a microsecond per evaluation
the budget evaporates fast - this motivates a heuristic such as QAOA.

In [ ]:
n_axis = np.arange(1, 61)
counts = 2.0 ** n_axis

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.semilogy(n_axis, counts, '-o', color=PALETTE['red'], lw=2, ms=5,
            label=r'$2^n$ bitstrings')

EVAL_PER_S = 1e6   # 1 microsecond per cost evaluation
bands = [
    ('1 sec',           EVAL_PER_S),
    ('1 hour',          3600 * EVAL_PER_S),
    ('1 day',           86400 * EVAL_PER_S),
    ('1 year',          365 * 86400 * EVAL_PER_S),
    ('age of universe', 1.4e10 * 365 * 86400 * EVAL_PER_S),
]
for label, count_at in bands:
    n_at = np.log2(count_at)
    ax.axvline(n_at, color=PALETTE['grey'], ls=':', lw=1)
    ax.text(n_at + 0.3, 1e3, f'{label}\n(n~{n_at:.0f})',
            color=PALETTE['charcoal'], fontsize=8, va='bottom')

ax.axvspan(0, np.log2(EVAL_PER_S), color=PALETTE['teal'], alpha=0.12,
           label='Brute force feasible (<= 1 s @ 1 us/eval)')

ax.set_xlabel('Number of assets n')
ax.set_ylabel('Solutions to enumerate')
title(ax, 'The combinatorial wall',
      '$2^n$ bitstrings to enumerate — why brute force fails as $n$ grows')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(0, 60)
ax.grid(False)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'combinatorial_wall.pdf', bbox_inches='tight')
plt.show()

### 4. THE COST LANDSCAPE QAOA TARGETS

For a fixed basket and budget K, every bitstring has a mean-variance cost
C(x) = -mu^T x + lambda x^T Sigma x + A (sum x - K)^2.
Sorting them ascending shows the search target: the leftmost red bar is the
global optimum brute force returns, and is what QAOA tries to concentrate
measurement probability on.

In [ ]:
K       = 2
LAM, AP = 2.0, 0.5
pf      = PortfolioProblem(mu, Sigma, lam=LAM, A=AP, K=K, tickers=tickers)

bits     = list(product([0, 1], repeat=n))
costs    = np.array([pf.cost(b) for b in bits])
feasible = np.array([sum(b) == K for b in bits])

order      = np.argsort(costs)
costs_s    = costs[order]
feasible_s = feasible[order]
labels_s   = [''.join(map(str, bits[i])) for i in order]

fig, ax = plt.subplots(figsize=FIGSIZE)
x    = np.arange(len(costs))
cols = [PALETTE['red'] if f else PALETTE['blue_muted'] for f in feasible_s]
ax.bar(x, costs_s, color=cols, edgecolor=PALETTE['charcoal'], linewidth=0.4)
ax.grid(False)
ax.set_xticks(x); ax.set_xticklabels(labels_s, rotation=90, fontsize=8)

gs_idx = int(np.argmin(np.where(feasible_s, costs_s, np.inf)))
ax.axvline(gs_idx, color=PALETTE['ochre'], ls='--', lw=1.5,
           label=f'Optimal K={K} portfolio: {labels_s[gs_idx]}  C={costs_s[gs_idx]:.4f}')
ax.set_xlabel('Bitstring (sorted by cost ascending)')
ax.set_ylabel(r'Portfolio cost $C(x) = -\mu^\top x + \lambda x^\top \Sigma x + A (\sum x - K)^2$')
title(ax, 'The cost landscape QAOA targets',
      f'All $2^{{{n}}}={2**n}$ bitstrings sorted ascending; red bars satisfy the budget K={K}')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'cost_landscape.pdf', bbox_inches='tight')
plt.show()